# Import and Shared functions

In [ ]:
import os, sys, re
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import importlib
import analyse_helpers
importlib.reload(analyse_helpers)
from analyse_helpers import resolve_log_files, load_kf_pid, load_pec

AXES   = ["M1", "M2", "M3"]        # Base-frame motor axes -- shared by every θ/ω field in both KFLOG and PIDLOG
TOPO   = ["Az", "Alt", "Roll"]     # Topocentric frame (α_*) axes
EQU    = ["RA", "Dec", "PA"]       # Equatorial frame (Δ_*) axes -- where issue #88 shows up
ARCSEC = 3600                      # deg -> arcsec, for readability at the scale this investigation cares about

PILOT_COLORS = {
    # Exact values from pilot/src/components/ChartXY.vue's `colors` object (used by both
    # AnalyseKalman.vue's Position/Velocity charts and AnalysePIDAll.vue) -- kept as the same
    # hsl(...) strings Pilot itself uses (plotly accepts CSS color strings directly), not
    # converted/approximated, so this is an exact match rather than a "close enough" one.
    "m1": "hsl(218, 63%, 32%)",   # Dark Blue -- Pilot reuses this one color for M1/M2/M3 alike
    "pv": "hsl(0, 0%, 100%)",     # White
    "sp": "hsl(132, 79%, 60%)",   # Green
    "op": "hsl(195, 99%, 70%)",   # Cyan
    "kp": "hsl(320, 70%, 30%)",   # Dark Magenta
    "ki": "hsl(50, 70%, 30%)",    # Dark Yellow ("olive")
    "kd": "hsl(20, 60%, 30%)",    # Dark Red ("brown")
    "ff": "hsl(132, 79%, 60%)",   # same value as sp in Pilot
}

def wrap_deg(x):
    """Wrap a degree value/difference into (-180, 180]. Every 'error' plot below is pv - sp,
    which needs this: without it, sp=359.9/pv=0.1 (e.g. Az crossing 0/360, or an RA/PA axis
    wrapping) gives a spurious -359.8 instead of the true +0.2."""
    return (x + 180) % 360 - 180


# Load data

In [ ]:
# ── Choose log path (last set log_filenames is what is used) ────────────────────────────────────────
# Single file:            'alpaca.log'                     (relative to LOG_DIR below)
# All rotated files:      'alpaca.log*'                    (glob -- handles a multi-hour run spanning several files)
# Explicit file list:     ['alpaca.log.2', 'alpaca.log.1', 'alpaca.log']
# A full/relative path (as before) also still works and bypasses LOG_DIR: '../logs/alpaca.log'
LOG_DIR = '../logs/archive'   # base directory log_filenames below are resolved against

log_filenames = ['alpaca.soak_nopec_Beta4.3_08_27a.log', 'alpaca.soak_nopec_Beta4.3_08_27b.log']
log_filenames = ['alpaca.soak_nopec_Beta4.3_08_28a.log']
log_filenames = ['alpaca.soak_nopec_Beta4.3_08_28k1.log','alpaca.soak_nopec_Beta4.3_08_28k2.log','alpaca.soak_nopec_Beta4.3_08_28k3.log','alpaca.soak_nopec_Beta4.3_08_28k4.log']
log_filenames = ['alpaca.soak_nopec_Beta4.3_08_29a.log']
log_filenames = ['alpaca.soak_nopec_Beta4.3_08_29b1.log','alpaca.soak_nopec_Beta4.3_08_29b2.log']
#log_filenames = ['alpaca.soak_nopec_Beta4.3_08_29n*.log']
#log_filenames = ['alpaca.soak_pec_Beta4.3_08_29m.log']                               # sync guiding, pec with log(kf,pos), half PEC on, half PEC off     66min
#log_filenames = ['alpaca.soak_nopec_Beta4.3_08_31*.log']
#log_filenames = ['alpaca.soak_Beta4.4_09_01_sg_sglog_Dec10_a*.log']                 # sync guiding,     with logs(sg), Eagle Nebula
#log_filenames = ['alpaca.soak_Beta4.4_09_03_sgpec_sgkfposlog_m83_a*.log']           # sync guiding,     with logs(sg), M83 Southern Pinwheel Galaxy     06:00PM - 09:30PM
#log_filenames = ['alpaca.soak_Beta4.4_09_03_sgpec_sglog_NGC55_b*.log']              # sync guiding, pec with logs(sg), NGC55 String of Pearls Galaxy    09:45PM - 11:45PM
#log_filenames = ['alpaca.soak_Beta4.4_09_03_sgpec_sgkfposlog_ngc2070_c*.log']       # sync guiding, pec with logs(sg,kf,pos), NGC2070 Tarantula Nebula  11:46PM - 02:30AM
log_filenames = ['alpaca.mark_Beta4.3_09_04_a*.log']                                # sync guiding      with logs(sg,kf,pos), 47 Tuc Marks log          2hr 47m session  

resolved_files = resolve_log_files(log_filenames, log_dir=LOG_DIR)
print(f"Loading {len(resolved_files)} file(s) from LOG_DIR={LOG_DIR!r}:")
for p in resolved_files:
    print(f"  {p}  ({os.path.getsize(p)/1e6:.1f} MB)")
print()

kf_df, pid_df = load_kf_pid(log_filenames, log_dir=LOG_DIR)
if not len(kf_df) and not len(pid_df):
    raise ValueError(f"No KFLOG/PIDLOG lines found in {resolved_files!r} -- was Config.log_position true during this session?")
print(f"KF  samples: {len(kf_df)}"  + (f"  ({kf_df.t_sec.iloc[-1]/60:.1f} min span)"  if len(kf_df)  else ""))
print(f"PID samples: {len(pid_df)}" + (f"  ({pid_df.t_sec.iloc[-1]/60:.1f} min span)" if len(pid_df) else ""))
if len(kf_df):
    print(f"KF  time range: {kf_df.timestamp.iloc[0]}  ->  {kf_df.timestamp.iloc[-1]}")
if len(pid_df):
    print(f"PID time range: {pid_df.timestamp.iloc[0]}  ->  {pid_df.timestamp.iloc[-1]}")
print()
print("KF columns:", list(kf_df.columns))
print("PID columns:", list(pid_df.columns))

# PECLOG (guide-sync events)

`PECLOG` (parsed the same way as `analyse_pec.ipynb`) is written once per guide-sync cycle
(`control.py:_pec_log()`, called from `process_guide_sync()`). Loaded here as `peclog_df` for
the sections below that need it as context: PEC-active/inactive transition markers on the KF
plot, the sync-guide zoom, and the PEC convergence check. Optional -- this capture only has
PECLOG entries if guiding was active during it.

In [ ]:
def load_peclog(log_filenames, log_dir='.'):
    """
    PECLOG entries used as context by later cells (PEC-active transition markers, sync-guide
    zoom, PEC convergence check), via the shared analyse_helpers.load_pec() (same PECLOG
    parsing as analyse_pec.ipynb). Returns an empty DataFrame (not an error) when none are
    found -- guiding/exposing isn't active in every capture, so PECLOG is optional/
    supplementary here (unlike in analyse_pec.ipynb, where it's the primary signal and
    load_pec() raising is exactly what's wanted).
    """
    try:
        df, _pec_config = load_pec(log_filenames, log_dir=log_dir)
    except ValueError:
        return pd.DataFrame()
    return df

peclog_df = load_peclog(log_filenames, log_dir=LOG_DIR)

if len(peclog_df):
    origin = kf_df.timestamp.iloc[0] if len(kf_df) else pid_df.timestamp.iloc[0]
    peclog_df["t_sec"] = (peclog_df.timestamp - origin).dt.total_seconds()
    print(f"PECLOG entries: {len(peclog_df)}")
else:
    print("No PECLOG entries in this capture.")

# Telemetry gaps (known 518-dropout artifact)

Per `docs/control.md`: the Polaris sometimes stops sending 518 telemetry for a few seconds,
during which `θ_pv` freezes while `θ_sp` keeps advancing -- producing a large but spurious
error spike once telemetry resumes, that looks like a control anomaly but isn't one. Flag
any gap much longer than the normal ~0.1-0.2s KFLOG cadence, before any plotting below, so
a brief total dropout can be masked out of every chart instead of blowing out its y-axis and
hiding the real, subtler variation those charts exist to show.

In [ ]:
GAP_THRESHOLD_SEC = 1.0   # normal cadence is ~0.1-0.2s; the known artifact runs a few seconds

if len(kf_df):
    gaps_df = kf_df[kf_df.gap_sec > GAP_THRESHOLD_SEC][["timestamp", "t_sec", "gap_sec"]].reset_index(drop=True)
    print(f"Flagged {len(gaps_df)} telemetry gap(s) > {GAP_THRESHOLD_SEC}s, "
          f"totalling {gaps_df.gap_sec.sum():.1f}s of lost telemetry")
else:
    gaps_df = pd.DataFrame(columns=["timestamp", "t_sec", "gap_sec"])
    print("No KF data loaded.")
gaps_df


# Gap masking (applies to every plot below)

Tags every KF/PID tick within a window around a flagged gap as `near_gap`. Every chart below
plots `series.mask(near_gap)` rather than the raw series -- pandas turns those ticks to NaN,
so plotly draws a visible break in the line instead of connecting through it or including it
in the y-axis autorange. The window covers the gap's *entire actual duration* (from
`gaps_df.gap_sec`, not a fixed constant -- some dropouts run over 100s, not a few seconds)
plus a small pad before it and a longer tail after it, since the KF/PID visibly takes tens of
seconds to resettle once telemetry resumes -- calibrated against a real run, widen/narrow
`GAP_EXCLUDE_BEFORE`/`GAP_EXCLUDE_AFTER` as needed.

In [ ]:
GAP_EXCLUDE_BEFORE = 2.0   # extra seconds before the last good sample to mask, as a safety pad
GAP_EXCLUDE_AFTER  = 30.0  # seconds after a gap to mask -- KF/PID visibly takes this long to resettle

def mask_near_gap(df, gaps_df, before_pad=GAP_EXCLUDE_BEFORE, after=GAP_EXCLUDE_AFTER):
    near = pd.Series(False, index=df.index)
    for _, g in gaps_df.iterrows():
        start = g.t_sec - g.gap_sec - before_pad   # back to the actual last good sample, not a fixed offset --
        end   = g.t_sec + after                     # a dropout can run well over 100s, not just a few
        near |= (df.t_sec >= start) & (df.t_sec <= end)
    return near

kf_df["near_gap"]  = mask_near_gap(kf_df, gaps_df)  if len(kf_df)  else pd.Series(dtype=bool)
pid_df["near_gap"] = mask_near_gap(pid_df, gaps_df) if len(pid_df) else pd.Series(dtype=bool)

if len(kf_df):
    print(f"KF  ticks masked as gap-affected: {kf_df.near_gap.sum()} / {len(kf_df)} ({kf_df.near_gap.mean()*100:.1f}%)")
if len(pid_df):
    print(f"PID ticks masked as gap-affected: {pid_df.near_gap.sum()} / {len(pid_df)} ({pid_df.near_gap.mean()*100:.1f}%)")


# Session Boundary Masking (startup + end transients -- applies to every plot below, same as gap masking)

A capture's first and last few seconds can each carry a one-off transient that isn't real
tracking behavior, distinct from a mid-session telemetry gap:

- **Startup** (existing): `theta_ref` isn't established until the first control cycle runs,
  so the earliest KFLOG tick(s) can carry raw absolute angles (seen directly in a real
  capture: M1/M2 `theta_meas`/`theta_state` over 100 degrees on the very first sample) instead
  of the near-zero reference-relative residual every later tick has, and PIDLOG's `Delta_sp`
  jumps by the full initial setpoint-establishment step in the first ~0.2s. Both settle within
  `STARTUP_EXCLUDE_SEC`.
- **End** (new): the mirror-image case -- a capture's last few seconds often coincide with
  tracking being deliberately stopped (goto/park/disconnect/`STOP tracking`), not a steady
  tracking state. `theta_ref` stops being populated once tracking is no longer active
  (confirmed against a real capture in `docs/pec_theta_space_plan.md`'s 08_31 analysis), and
  the setpoint/PID state right around that transition isn't representative of ordinary
  tracking either. Unlike Startup, this hasn't been empirically confirmed the same way against
  a real capture yet -- `END_EXCLUDE_SEC` is a precautionary, symmetric default; set it to 0
  for a given capture if its tail turns out to already be clean.

Masked the same way as `near_gap` (as `near_startup`/`near_end`), and combined into one
`exclude` column used everywhere below -- so every plot/metric excludes both consistently
instead of each cell reinventing its own threshold (this is exactly what happened before: the
RA/PA anomaly hunt further down had its own local startup exclusion, but the KF plots/
variation-reduction report above it didn't).

In [ ]:
STARTUP_EXCLUDE_SEC = 30.0   # seconds from the start of the run to mask as startup transient
END_EXCLUDE_SEC     = 30.0   # seconds from the end of the run to mask as shutdown/stop-tracking transient

kf_df["near_startup"]  = (kf_df.t_sec  < STARTUP_EXCLUDE_SEC) if len(kf_df)  else pd.Series(dtype=bool)
pid_df["near_startup"] = (pid_df.t_sec < STARTUP_EXCLUDE_SEC) if len(pid_df) else pd.Series(dtype=bool)

kf_df["near_end"]  = (kf_df.t_sec  > kf_df.t_sec.max()  - END_EXCLUDE_SEC) if len(kf_df)  else pd.Series(dtype=bool)
pid_df["near_end"] = (pid_df.t_sec > pid_df.t_sec.max() - END_EXCLUDE_SEC) if len(pid_df) else pd.Series(dtype=bool)

kf_df["exclude"]  = (kf_df.near_gap  | kf_df.near_startup  | kf_df.near_end)  if len(kf_df)  else pd.Series(dtype=bool)
pid_df["exclude"] = (pid_df.near_gap | pid_df.near_startup | pid_df.near_end) if len(pid_df) else pd.Series(dtype=bool)

if len(kf_df):
    print(f"KF  ticks masked as startup transient: {kf_df.near_startup.sum()} / {len(kf_df)} "
          f"({kf_df.near_startup.mean()*100:.1f}%)")
    print(f"KF  ticks masked as end transient:     {kf_df.near_end.sum()} / {len(kf_df)} "
          f"({kf_df.near_end.mean()*100:.1f}%)")
if len(pid_df):
    print(f"PID ticks masked as startup transient: {pid_df.near_startup.sum()} / {len(pid_df)} "
          f"({pid_df.near_startup.mean()*100:.1f}%)")
    print(f"PID ticks masked as end transient:     {pid_df.near_end.sum()} / {len(pid_df)} "
          f"({pid_df.near_end.mean()*100:.1f}%)")

# KF: Measured vs Filtered, position already reference-relative, velocity vs ω_ref

`θ_meas`/`θ_state` already have `theta_ref` subtracted by the driver whenever tracking is
enabled (`control.py`'s `observe()` passes `theta_ref=self._pid.theta_ref` into
`wrap_angle_residual()`) -- plotting them directly, with no further subtraction, is correct.
`ω_meas`/`ω_state` are different: KFLOG logs those completely raw
(`omega_meas.flatten().tolist()`, never passed through `wrap_angle_residual`), so comparing
them to `ω_ref` (also a KFLOG field) has to happen here in the notebook, not in the driver.

In [ ]:
fig = make_subplots(rows=3, cols=2, shared_xaxes=True, 
    subplot_titles=sum([[f"{ax} -- θ_meas vs θ_state (arcsec, already ref-relative)", f"{ax} -- (ω_meas - ω_ref) vs (ω_state - ω_ref) (arcsec/s)"] for ax in AXES], []),
    vertical_spacing=0.06)
exclude = kf_df.exclude

# pec_active transitions (False->True and True->False), marked on the position column so it's
# visible whether a change in chop/oscillation character lines up with PEC actually engaging
# or disengaging, rather than just eyeballing timestamps against the PECLOG cells below.
pec_transitions = []
if len(peclog_df):
    active = peclog_df.pec_active.astype(bool)
    changed = active.ne(active.shift()).fillna(True)
    for t, is_active in zip(peclog_df.t_sec[changed], active[changed]):
        pec_transitions.append((t, is_active))

for i, ax in enumerate(AXES):
    row = i + 1
    meas_err  = (kf_df[f"θ_meas_{row}"]  * ARCSEC).mask(exclude)
    state_err = (kf_df[f"θ_state_{row}"] * ARCSEC).mask(exclude)
    fig.add_trace(go.Scatter(x=kf_df.timestamp, y=meas_err, name=f"{ax} θ_meas",
        line=dict(color=PILOT_COLORS["m1"], width=1)), row=row, col=1)
    fig.add_trace(go.Scatter(x=kf_df.timestamp, y=state_err, name=f"{ax} θ_state",
        line=dict(color=PILOT_COLORS["pv"], width=1)), row=row, col=1)
    # for t, is_active in pec_transitions:
    #     fig.add_vline(x=t, line=dict(color="lime" if is_active else "red", width=1, dash="dash"),
    #         row=row, col=1)
    meas_verr  = ((kf_df[f"ω_meas_{row}"]  - kf_df[f"ω_ref_{row}"]) * ARCSEC).mask(exclude)
    state_verr = ((kf_df[f"ω_state_{row}"] - kf_df[f"ω_ref_{row}"]) * ARCSEC).mask(exclude)
    fig.add_trace(go.Scatter(x=kf_df.timestamp, y=meas_verr, name=f"{ax} ω_meas-ω_ref",
        line=dict(color=PILOT_COLORS["m1"], width=1)), row=row, col=2)
    fig.add_trace(go.Scatter(x=kf_df.timestamp, y=state_verr, name=f"{ax} ω_state-ω_ref",
        line=dict(color=PILOT_COLORS["pv"], width=1)), row=row, col=2)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)
fig.update_xaxes(title_text="Time (s)", row=3, col=2)
fig.update_xaxes(matches="x")   # link every subplot's x-axis to the first, across both columns
fig.update_layout(height=900, width=1500, template="plotly_dark", hovermode="x unified",
    title="Kalman Filter -- Measured vs Filtered, relative to reference "
          "(dashed green/red = PEC became active/inactive)",
    legend=dict(groupclick="toggleitem"))
fig.show()


# KF: variation reduced vs the raw input signal

Reports `std(θ_state)` against `std(θ_meas)` per axis (and the equivalent against `ω_ref`
for `ω`, see markdown above) as a direct "how much noise did the filter remove" number --
gap-affected ticks excluded, since a few huge dropout-recovery spikes would otherwise
dominate a std() and make the filter look far more/less effective than it actually is on
ordinary ticks.

A large std here doesn't necessarily mean noise, though -- it's just as consistent with a
real, systematic bias or drift (constant offset, or a steady trend over the session) sitting
underneath. Since std alone can't tell those apart, this also reports mean and a linear
time-trend per axis: a mean far from zero with a *small* trend looks like a persistent bias;
a small mean with a large trend looks like something drifting away over the session (e.g. an
uncorrected/mis-signed periodic or secular term); large std with both small is closer to
genuine noise.

In [ ]:
clean_kf = kf_df[~kf_df.exclude] if len(kf_df) else kf_df

SANE_STD_ARCSEC = 30.0   # flag axes whose std is this large -- worth characterizing further below

def report_reduction(label, unit, meas, state):
    raw_std  = meas.std()
    filt_std = state.std()
    pct = (1 - filt_std / raw_std) * 100 if raw_std > 0 else float("nan")
    word = "reduction" if pct >= 0 else "increase"
    flag = f"  <-- std > {SANE_STD_ARCSEC:.0f}, see characterization below" if raw_std > SANE_STD_ARCSEC else ""
    print(f"  {label}: raw std={raw_std:9.3f}{unit} -> filtered std={filt_std:9.3f}{unit}  ({pct:+.1f}% {word}){flag}")

def characterize(label, series, t_sec):
    slope = np.polyfit(t_sec, series, 1)[0] * 3600  # arcsec/hr
    half = t_sec.min() + (t_sec.max() - t_sec.min()) / 2
    first_half_mean  = series[t_sec < half].mean()
    second_half_mean = series[t_sec >= half].mean()
    print(f"    {label}: mean={series.mean():+9.2f}\"  trend={slope:+9.2f}\"/hr  "
          f"1st-half mean={first_half_mean:+9.2f}\"  2nd-half mean={second_half_mean:+9.2f}\"")

if len(clean_kf):
    print("Position (θ, arcsec):")
    large_axes = []
    for i, ax in enumerate(AXES):
        meas  = clean_kf[f"θ_meas_{i+1}"]  * ARCSEC
        state = clean_kf[f"θ_state_{i+1}"] * ARCSEC
        report_reduction(ax, '"', meas, state)
        if meas.std() > SANE_STD_ARCSEC:
            large_axes.append((ax, meas))

    print("\nVelocity (ω - ω_ref, arcsec/s):")
    for i, ax in enumerate(AXES):
        meas_verr  = (clean_kf[f"ω_meas_{i+1}"]  - clean_kf[f"ω_ref_{i+1}"]) * ARCSEC
        state_verr = (clean_kf[f"ω_state_{i+1}"] - clean_kf[f"ω_ref_{i+1}"]) * ARCSEC
        report_reduction(ax, '"/s', meas_verr, state_verr)

    if large_axes:
        print(f"\nCharacterizing axes with std > {SANE_STD_ARCSEC:.0f}\" (bias vs. drift vs. noise):")
        for ax, meas in large_axes:
            characterize(ax, meas, clean_kf.t_sec)
else:
    print("No clean KF data to compute a reduction metric from.")

# KF: Gain per axis (position + velocity)

`K_gain` is the diagonal of the Kalman gain matrix: indices 1-3 are the position gains
(θ1-3), 4-6 the velocity gains (ω1-3). A gain spike means the filter suddenly started
trusting a raw measurement much more than usual -- worth cross-checking against any
θ_meas/θ_state divergence at the same tick.

In [ ]:
fig = go.Figure()
labels = [f"{ax} pos" for ax in AXES] + [f"{ax} vel" for ax in AXES]
colors = ["royalblue", "orange", "mediumseagreen", "royalblue", "orange", "mediumseagreen"]
dashes = ["solid", "solid", "solid", "dash", "dash", "dash"]
for i, (label, color, dash) in enumerate(zip(labels, colors, dashes)):
    fig.add_trace(go.Scatter(x=kf_df.t_sec, y=kf_df[f"K_gain_{i+1}"].mask(kf_df.exclude), name=label,
        line=dict(color=color, width=1, dash=dash)))
fig.update_layout(height=500, width=1300, template="plotly_dark", hovermode="x unified",
    title="Kalman Gain per axis", xaxis_title="Time (s)", yaxis_title="Gain",
    legend=dict(groupclick="toggleitem"))
fig.show()


# PID: Position tracking error (θ, Base-frame motor angles)

Plotted as `θ_pv - θ_sp` (arcsec, wrap-safe -- see `wrap_deg`) rather than the two raw lines
overlaid -- M1-M3 drift together as the mount tracks, so an SP-vs-PV overlay is two
near-identical slowly-moving lines with the actual tracking error invisible at that scale.
This is the error directly.

In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=[f"{ax} -- θ_pv - θ_sp (arcsec)" for ax in AXES], vertical_spacing=0.08)
for i, ax in enumerate(AXES):
    row = i + 1
    error = (wrap_deg(pid_df[f"θ_pv_{row}"] - pid_df[f"θ_sp_{row}"]) * ARCSEC).mask(pid_df.exclude)
    fig.add_trace(go.Scatter(x=pid_df.t_sec, y=error, name=f"{ax} error",
        line=dict(color="white", width=1)), row=row, col=1)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)
fig.update_layout(height=800, width=1300, template="plotly_dark", hovermode="x unified",
    title="PID Position Tracking Error", legend=dict(groupclick="toggleitem"))
fig.show()


# KF vs PID: frame-mismatch check (SGC/MAC/PGC not applied to θ_meas_raw)

`θ_meas`/`θ_state` (KFLOG) are diffed against `theta_ref` inside `control.py`'s
`KalmanFilter.observe()` (display only -- not fed into the filter). But `theta_ref`
(`self._pid.theta_ref`, same value as PIDLOG's `θ_sp`) is derived via `topoQ_to_baseQ()`,
which *deliberately does not undo* Mechanical Alignment (MAC), Sync Guide (SGC), or Pulse
Guide (PGC) corrections -- the comment there is explicit: "PID theta works in corrected theta
space". Meanwhile the raw measurement fed into `KF.observe()` (`theta_raw` in `polaris.py`'s
`decode_518position_measurement()`) comes straight from the 518 message's quaternion with
NONE of those corrections applied.

So every time a sync guide lands (SGC updates `q_syncguide_B`) or the mount moves through the
sky far enough for MAC's correction to change, `theta_ref` shifts in "corrected" space while
the raw measurement doesn't -- producing exactly a chop (discrete SGC update) or a slow ramp
(continuously-varying MAC) in the KF's ref-relative residual, independent of whether the mount
is actually tracking well.

`θ_pv` (PIDLOG) is NOT affected by this -- it's built via `baseQ_to_topoQ()`, which applies
the same MAC+SGC+PGC chain that `theta_ref` implicitly includes, so `θ_pv - θ_sp` (already
plotted above, "PID Position Tracking Error") is frame-matched and should reflect real
tracking error only.

If the first chart below (raw `θ_meas_raw` vs `θ_ref_raw`, absolute degrees, no ref
subtraction -- your original ask) shows discrete jumps or a ramp lining up with the KFLOG
chop/ramp above, and the second chart shows the KFLOG residual tracking a similar shape while
the PIDLOG residual stays small and clean, that confirms this is a display/telemetry framing
issue in KFLOG, not a real physical mistracking problem.

Separately, `ω_pec` (already in PIDLOG) is overlaid on the second chart too: PEC's own
periodic rate correction is applied to the motor (`omega_tgt = ... + omega_ff - omega_pec` in
`control.py`) but is never folded back into `theta_ref`'s position target either, which would
independently produce a residual sine matching PEC's `applied_rate` period even with zero
frame-mismatch. Sign convention for the cumulative overlay hasn't been empirically verified --
compare shape/period first, flip the sign in the code below if it comes out inverted.


In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=[f"{ax} -- raw θ_meas_raw vs θ_ref_raw (deg, absolute, NOT ref-relative)" for ax in AXES],
    vertical_spacing=0.08)
for i, ax in enumerate(AXES):
    row = i + 1
    meas_raw = kf_df[f"θ_meas_raw_{row}"].mask(kf_df.exclude)
    fig.add_trace(go.Scatter(x=kf_df.t_sec, y=meas_raw, name=f"{ax} θ_meas_raw",
        line=dict(color=PILOT_COLORS["m1"], width=1)), row=row, col=1)
    ref_col = f"θ_ref_raw_{row}"
    if ref_col in kf_df.columns:
        fig.add_trace(go.Scatter(x=kf_df.t_sec, y=kf_df[ref_col].mask(kf_df.exclude), name=f"{ax} θ_ref_raw",
            line=dict(color=PILOT_COLORS["sp"], width=1)), row=row, col=1)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)
fig.update_layout(height=800, width=1300, template="plotly_dark", hovermode="x unified",
    title="KF Raw Absolute Position (pre-backdate, no ref subtraction) -- gaps = tracking was off (theta_ref was None)",
    legend=dict(groupclick="toggleitem"))
fig.show()

# Direct frame-mismatch test: KFLOG's ref-relative residual (raw meas vs corrected-space ref,
# as currently logged) vs PIDLOG's already frame-matched θ_pv - θ_sp, same axes/scale/units,
# overlaid -- plus a cumulative ω_pec trace to test the separate PEC-not-in-theta_ref hunch.
fig2 = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=[f"{ax} -- KFLOG θ_meas (raw vs corrected ref) vs PIDLOG θ_pv-θ_sp (frame-matched) vs cumulative ω_pec" for ax in AXES],
    vertical_spacing=0.08)
for i, ax in enumerate(AXES):
    row = i + 1
    kf_delta  = (kf_df[f"θ_meas_{row}"] * ARCSEC).mask(kf_df.exclude)
    pid_delta = (wrap_deg(pid_df[f"θ_pv_{row}"] - pid_df[f"θ_sp_{row}"]) * ARCSEC).mask(pid_df.exclude)
    fig2.add_trace(go.Scatter(x=kf_df.t_sec, y=kf_delta, name=f"{ax} KFLOG θ_meas",
        line=dict(color=PILOT_COLORS["m1"], width=1)), row=row, col=1)
    fig2.add_trace(go.Scatter(x=pid_df.t_sec, y=pid_delta, name=f"{ax} PIDLOG θ_pv-θ_sp",
        line=dict(color="orange", width=1)), row=row, col=1)
    pec_col = f"ω_pec_{row}"
    if pec_col in pid_df.columns:
        dt = pid_df.t_sec.diff().fillna(0)
        pec_integral = -(pid_df[pec_col].fillna(0) * dt).cumsum() * ARCSEC   # sign: omega_tgt = ... + omega_ff - omega_pec
        fig2.add_trace(go.Scatter(x=pid_df.t_sec, y=pec_integral.mask(pid_df.exclude), name=f"{ax} -cumulative ω_pec",
            line=dict(color="mediumseagreen", width=1, dash="dot")), row=row, col=1)
fig2.update_xaxes(title_text="Time (s)", row=3, col=1)
fig2.update_layout(height=900, width=1300, template="plotly_dark", hovermode="x unified",
    title="Frame-mismatch check: KF (raw vs corrected ref) vs PID (frame-matched) vs cumulative PEC rate",
    legend=dict(groupclick="toggleitem"))
fig2.show()


# Does flipping ω_pec's sign for M1/M3 actually cancel the ramp between syncs?

Direct visual test of the "if I flipped M1's PEC direction, the sync-guide steps would
disappear" observation. Both `θ_meas` (which already includes the actual applied `omega_pec`,
however it's signed) and cumulative `ω_pec` for the SAME axis and window are detrended to
start at 0, so only the *shape/slope* between syncs is being compared, not any constant
offset from the frame mismatch above.

Two candidate overlays are plotted: `+cumulative(ω_pec)` and `-cumulative(ω_pec)`. Whichever
one visually cancels the ramp in `θ_meas` (i.e. `θ_meas` minus that overlay is flat between
syncs, only stepping at the sync events) is the one that matches what's actually happening on
that axis.

Important: this does **not** contradict the steady-state Kp/Ki/Kd verification, which showed
`omega_tgt = ... - omega_pec` is correct overall (checked directly against the converged
closed-loop terms, at 3 orientations). `ω_pec` is a **Base-frame** RA/Dec quantity
(`omega_pec_B` in `control.py`) that only becomes a *per-motor* M1/M2/M3 rate after
`feed_forward()` solves it through `theta_to_jacobian(*theta_pv)` -- a matrix that depends on
the mount's current orientation. If this shows a clean sign flip specifically on M1/M3 (not
M2) at this orientation, the bug isn't `omega_tgt`'s formula -- it's downstream of it, in how
that correctly-signed Base-frame correction gets distributed across the three motors here.


In [ ]:
ZOOM_T0, ZOOM_T1 = 500, 1500   # seconds -- the window where the ramp/chop is visible
ZOOM_AXES = ["M1", "M3"]        # the axes reported as backwards; M2 is the control (clean)

fig = make_subplots(rows=len(ZOOM_AXES), cols=1, shared_xaxes=True,
    subplot_titles=[f"{ax} -- θ_meas vs ±cumulative ω_pec (both detrended to 0 at window start)"
                     for ax in ZOOM_AXES],
    vertical_spacing=0.1)

kf_zoom  = kf_df[(kf_df.t_sec >= ZOOM_T0) & (kf_df.t_sec <= ZOOM_T1)]
pid_zoom = pid_df[(pid_df.t_sec >= ZOOM_T0) & (pid_df.t_sec <= ZOOM_T1)]

for r, ax in enumerate(ZOOM_AXES):
    row = r + 1
    i = AXES.index(ax) + 1

    meas = (kf_zoom[f"θ_meas_{i}"] * ARCSEC).mask(kf_zoom.exclude)
    meas_detrend = meas - meas.iloc[0]
    fig.add_trace(go.Scatter(x=kf_zoom.t_sec, y=meas_detrend, name=f"{ax} θ_meas (detrended)",
        line=dict(color="white", width=2)), row=row, col=1)

    dt = pid_zoom.t_sec.diff().fillna(0)
    pec_col = f"ω_pec_{i}"
    if pec_col in pid_zoom.columns:
        raw_integral = (pid_zoom[pec_col].fillna(0) * dt).cumsum() * ARCSEC
        for sign, label, color in [(+1, "+cumulative ω_pec", "orange"), (-1, "-cumulative ω_pec", "mediumseagreen")]:
            overlay = sign * raw_integral
            overlay = overlay - overlay.iloc[0]
            fig.add_trace(go.Scatter(x=pid_zoom.t_sec, y=overlay, name=f"{ax} {label}",
                line=dict(color=color, width=1, dash="dot")), row=row, col=1)

fig.update_xaxes(title_text="Time (s)", row=len(ZOOM_AXES), col=1)
fig.update_layout(height=350 * len(ZOOM_AXES) + 100, width=1300, template="plotly_dark",
    hovermode="x unified",
    title=f"[{ZOOM_T0},{ZOOM_T1}]s -- whichever cumulative ω_pec sign flattens θ_meas between syncs is the real per-axis sign",
    legend=dict(groupclick="toggleitem"))
fig.show()


# PEC sign check: zoom into individual sync-guide events

Two competing hunches for the M1/M3 sawtooth (M2 looks clean): (a) `omega_pec`'s sign in
`omega_tgt` is backwards (see commit history below), or (b) the PEC model itself was just
poorly converged/noisy over this period, independent of sign. This section is built to tell
them apart from the actual data, not from more code-reading.

**Sign trace (from the code + the project's own proven axis tests in
`tests/test_pole_axes_B.py`):**
- `process_guide_sync()` defines `ra_resid = observed_RA - believed_RA` (positive = truth
  is further along in +RA than we currently think).
- `test_ra_axis_matches_increasing_ra_at_fixed_time` proves `+RA == +rotation about
  ra_axis_B` (no negation) at fixed time. So `accumulate_sync_guiding_residuals()`'s
  `Quaternion(axis=ra_axis_B, degrees=+ra_resid)`, pre-multiplied onto the motor quaternion
  (the same composition convention `test_alpha_axis_signs_random_sweep` verifies elsewhere),
  correctly nudges the PV *toward* the observed truth. **PV/SGC side checks out.**
- PEC's `d_ra` (`eval_correction`) is fit directly on that same `ra_resid` history, so it
  inherits the same sign meaning: positive `d_ra` means the mount needs to move *more* in
  +RA to keep up. `omega_pec_B = +(d_ra/dt) * ra_axis_B` (`apply_pec_drift_correction`,
  unchanged since it was introduced) is therefore already a "+RA-direction" extra rate by
  the same proven convention -- which means it should be **added** to `omega_tgt`, matching
  the original `0813ca0c` ("!!!CRITICAL FIX!!!") design and its explicit "lockstep" rationale
  (still sitting in `apply_pec_drift_correction`'s docstring, unchanged).
- `2683d6d` flipped `omega_tgt` to `- omega_pec` a day after a telemetry-only sign flip in
  `f87a0b91`, without touching `accumulate_sync_guiding_residuals()` or that docstring.

**Why "Ki disappeared" isn't proof either way:** Ki calming down only shows PV and FF became
*paired* consistently with each other (SP-PV stayed flat) -- it can't distinguish "correctly
paired in the direction that tracks the real sky" from "consistently paired in the *wrong*
direction, with the control loop internally happy while genuinely drifting between syncs."
The second case is exactly a sign-inverted-but-confident model, and would show up as a
*textbook-clean* sawtooth: a consistent ramp, same direction and similar size every cycle,
snapped back at each sync -- not noisy/inconsistent the way a genuinely poorly-converged
model would look.

**The actual discriminator, run on your data below:** for each sync-guide event, take the
slope of `θ_meas_raw` in the ~25s immediately before it.
- **Consistent slope (same sign, similar magnitude, sync after sync)** → sign bug, not a
  model-quality problem -- a bad/noisy fit would not produce a clean repeating ramp.
- **Inconsistent slope (flips sign, magnitude varies a lot) alongside low `r2`** → genuinely
  supports the "PEC model was just terrible over this period" hunch instead.

Also check `r2_1`/`r2_2` (RA/Dec fit quality) printed per event: if RA's r2 was solidly high
while M1/M3 still sawtooth, that argues *against* "terrible model" for RA specifically (a
good fit applied backwards still looks clean-and-wrong). M2 being clean while M1/M3 aren't is
also consistent with a sign bug isolated to the RA-axis contribution: PEC drives `omega_pec_B`
as a combination of `ra_axis_B` and `dec_axis_B`, solved through the full 3-axis Jacobian --
depending on current mount geometry, RA's correction can couple mostly into M1/M3 while Dec's
couples mostly into M2, so a RA-only sign issue would selectively show up exactly where you're
seeing it.


In [ ]:
# ── Zoom into a few real sync-guide events ──────────────────────────────────────
WINDOW_T0, WINDOW_T1 = 940, 1060   # seconds -- adjust to bracket the syncs you want to inspect

pec_window = peclog_df[(peclog_df.t_sec >= WINDOW_T0) & (peclog_df.t_sec <= WINDOW_T1)]
print(f"{len(pec_window)} sync-guide event(s) in [{WINDOW_T0}, {WINDOW_T1}]s:")
for _, ev in pec_window.iterrows():
    print(f"  t={ev.t_sec:7.1f}s  n={ev.n:.0f}  "
          f"resid(RA,Dec)=({ev.resid_1:+7.3f},{ev.resid_2:+7.3f})'  "
          f"applied_rate(RA,Dec)=({ev.applied_rate_1:+8.2f},{ev.applied_rate_2:+8.2f})'/hr  "
          f"r2(RA,Dec)=({ev.r2_1:.2f},{ev.r2_2:.2f})  pec_active={ev.pec_active}")

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=[f"{ax} -- θ_meas_raw (deg) with sync-guide events marked (red)" for ax in AXES],
    vertical_spacing=0.08)
kf_win = kf_df[(kf_df.t_sec >= WINDOW_T0) & (kf_df.t_sec <= WINDOW_T1)]
for i, ax in enumerate(AXES):
    row = i + 1
    fig.add_trace(go.Scatter(x=kf_win.t_sec, y=kf_win[f"θ_meas_raw_{row}"], name=f"{ax} θ_meas_raw",
        line=dict(color=PILOT_COLORS["m1"], width=1)), row=row, col=1)
    for _, ev in pec_window.iterrows():
        fig.add_vline(x=ev.t_sec, line=dict(color="red", width=1, dash="dot"), row=row, col=1)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)
fig.update_layout(height=800, width=1300, template="plotly_dark", hovermode="x unified",
    title=f"Sync-guide zoom [{WINDOW_T0},{WINDOW_T1}]s -- red dashed = sync-guide event",
    legend=dict(groupclick="toggleitem"))
fig.show()

# ── Ramp-consistency check across every sync while PEC is active ───────────────────────────
# Slope of θ_meas_raw in the PRE_WINDOW_S immediately before each sync: consistent sign/size
# sync after sync => sign bug; inconsistent => genuinely noisy/unconverged model.
PRE_WINDOW_S = 25.0
records = []
for _, ev in peclog_df[peclog_df.pec_active].iterrows():
    seg = kf_df[(kf_df.t_sec >= ev.t_sec - PRE_WINDOW_S) & (kf_df.t_sec < ev.t_sec)]
    if len(seg) < 5:
        continue
    for i, ax in enumerate(AXES):
        slope = np.polyfit(seg.t_sec, seg[f"θ_meas_raw_{i+1}"], 1)[0] * 3600  # arcsec/s
        records.append({"t_sec": ev.t_sec, "axis": ax, "pre_slope_arcsec_s": slope,
                         "resid_ra": ev.resid_1, "resid_dec": ev.resid_2,
                         "r2_ra": ev.r2_1, "r2_dec": ev.r2_2})

ramp_df = pd.DataFrame(records)
if len(ramp_df):
    print("\nPre-sync θ_meas_raw slope (arcsec/s) by axis, per sync event:")
    print(ramp_df.pivot_table(index="t_sec", columns="axis", values="pre_slope_arcsec_s").round(2))
    print()
    for ax in AXES:
        vals = ramp_df.loc[ramp_df.axis == ax, "pre_slope_arcsec_s"]
        same_sign_frac = max((vals > 0).mean(), (vals < 0).mean())
        print(f"{ax}: mean={vals.mean():+7.2f}\"/s  std={vals.std():6.2f}\"/s  "
              f"same-sign-as-majority={same_sign_frac*100:4.1f}% of {len(vals)} syncs "
              f"({'looks like a consistent (sign?) bias' if same_sign_frac > 0.8 else 'looks noisy/inconsistent'})")
else:
    print("No PEC-active sync events with enough KF data before them in this session.")


# Is PEC's correction actually shrinking the raw guide error over the session?

The chop in the KFLOG chart *is* each sync's `resid` being folded into `q_syncguide_B` (see
frame-mismatch section above) -- so if it's still choppy after ~1000s+ of PEC tuning, that's
not just a display artifact anymore: PECLOG's own `resid` field is documented as "this sync's
raw guide error -- should shrink as PEC improves" (`control.py:2732`). If it isn't shrinking
despite `pec_active=True` and a decent `r2`, PEC is not actually converging on that axis,
independent of the omega_tgt sign question above (which the steady-state Kp/Ki/Kd test already
settled).

Plotted below: `resid` magnitude (RA, Dec) at every sync across the whole session, a rolling
RMS trend to see through sync-to-sync scatter, and `r2`/`pec_active` underneath so a
non-shrinking trend can be told apart from "PEC simply hadn't gone active yet" or "the fit
quality was never good enough to trust."


In [ ]:
ROLL_N = 8   # syncs -- rolling RMS window for the resid trend line

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, row_heights=[0.45, 0.45, 0.1],
    subplot_titles=["RA resid (arcmin) -- should shrink as PEC improves",
                     "Dec resid (arcmin) -- should shrink as PEC improves",
                     "Fit quality (r2) and PEC active"],
    vertical_spacing=0.06)

for col, label, color in [("resid_1", "RA", "royalblue"), ("resid_2", "Dec", "orange")]:
    row = 1 if label == "RA" else 2
    resid_abs = peclog_df[col].abs()
    rms = (peclog_df[col] ** 2).rolling(ROLL_N, min_periods=3).mean().pow(0.5)
    fig.add_trace(go.Scatter(x=peclog_df.t_sec, y=resid_abs, mode="markers", name=f"{label} |resid|",
        marker=dict(color=color, size=5, opacity=0.5)), row=row, col=1)
    fig.add_trace(go.Scatter(x=peclog_df.t_sec, y=rms, name=f"{label} rolling RMS ({ROLL_N} syncs)",
        line=dict(color="white", width=2)), row=row, col=1)
    fig.update_yaxes(title_text="arcmin", row=row, col=1)

fig.add_trace(go.Scatter(x=peclog_df.t_sec, y=peclog_df.r2_1, name="r2 RA",
    line=dict(color="royalblue", width=1)), row=3, col=1)
fig.add_trace(go.Scatter(x=peclog_df.t_sec, y=peclog_df.r2_2, name="r2 Dec",
    line=dict(color="orange", width=1)), row=3, col=1)
fig.add_trace(go.Scatter(x=peclog_df.t_sec, y=peclog_df.pec_active.astype(int), name="pec_active",
    line=dict(color="mediumseagreen", width=1, dash="dot")), row=3, col=1)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)
fig.update_layout(height=750, width=1300, template="plotly_dark", hovermode="x unified",
    title="PEC convergence check: is the raw guide error actually shrinking over the session?",
    legend=dict(groupclick="toggleitem"))
fig.show()

# Quantify it directly: first-N vs last-N syncs (while pec_active), RA and Dec.
N_COMPARE = 10
active = peclog_df[peclog_df.pec_active]
if len(active) >= 2 * N_COMPARE:
    for col, label in [("resid_1", "RA"), ("resid_2", "Dec")]:
        first_rms = (active[col].iloc[:N_COMPARE] ** 2).mean() ** 0.5
        last_rms  = (active[col].iloc[-N_COMPARE:] ** 2).mean() ** 0.5
        pct = (1 - last_rms / first_rms) * 100 if first_rms > 0 else float("nan")
        verdict = "shrinking (PEC improving)" if pct > 20 else "flat/growing (PEC NOT improving)" if pct < -5 else "roughly unchanged"
        print(f"{label}: first {N_COMPARE} active syncs RMS={first_rms:.3f}' -> "
              f"last {N_COMPARE} RMS={last_rms:.3f}'  ({pct:+.0f}%)  -- {verdict}")
else:
    print(f"Fewer than {2*N_COMPARE} PEC-active syncs this session -- not enough to compare first vs last reliably.")


# PID: Velocity breakdown (Kp / Ki / Kd / FF / Output)

Matches the Alpaca Pilot PID Tuning page convention: Cyan = Output (ω_op, what actually
drove the motor), Magenta = Kp, Olive = Ki, Orange = Kd, Green = FF (currently ω_ff − ω_pec
combined -- PEC is not yet broken out as its own field in the payload).

In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=[f"{ax} -- velocity components (arcsec/s)" for ax in AXES], vertical_spacing=0.08)
components = [("ω_ff", PILOT_COLORS["ff"]), ("ω_kp", PILOT_COLORS["kp"]), ("ω_ki", PILOT_COLORS["ki"]), ("ω_kd", PILOT_COLORS["kd"]), ("ω_op", PILOT_COLORS["op"])]
for i, ax in enumerate(AXES):
    row = i + 1
    for key, color in components:
        fig.add_trace(go.Scatter(x=pid_df.t_sec, y=(pid_df[f"{key}_{row}"] * ARCSEC).mask(pid_df.exclude), name=f"{ax} {key}",
            line=dict(color=color, width=1)), row=row, col=1)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)
fig.update_layout(height=900, width=1300, template="plotly_dark", hovermode="x unified",
    title="PID Velocity Breakdown", legend=dict(groupclick="toggleitem"))
fig.show()


# PID: Equatorial & Topocentric tracking error (RA/Dec/PA, Az/Alt/Roll)

Issue #88's reported symptom (near-meridian RA/PA noise) shows up in the **equatorial**
frame (`Δ_*`), not the base motor frame (`θ_*`) plotted above -- M1/M2/M3 look clean even
during an affected period, so this view is the one that actually matters for that
investigation. Both columns plot `pv - sp` (error, wrap-safe -- see `wrap_deg`) rather than
an SP/PV overlay. RA/Dec need this because they barely move during single-target tracking, so
the raw lines would overlap and hide the error -- but Az/Alt/Roll need it too, for a different
reason: they *do* move a lot over a session (tens of degrees), so the actual tracking error
(arcsec-scale) would be completely invisible against that range on a raw overlay, and Az
crossing 0°/360° would otherwise show as a spurious ~360° spike without the wrap-safe diff.
`α_pv_1` = Az, marked against meridian transit (Az≈178-180°) with a dotted line for reference.

In [ ]:
fig = make_subplots(rows=3, cols=2, shared_xaxes=True,
    subplot_titles=sum([[f"{ax} -- Δ_pv - Δ_sp (arcsec)", f"{TOPO[i]} -- α_pv - α_sp (arcsec)"] for i, ax in enumerate(EQU)], []),
    vertical_spacing=0.06)
exclude = pid_df.exclude
for i in range(3):
    row = i + 1
    equ_error  = (wrap_deg(pid_df[f"Δ_pv_{row}"] - pid_df[f"Δ_sp_{row}"]) * ARCSEC).mask(exclude)
    topo_error = (wrap_deg(pid_df[f"α_pv_{row}"] - pid_df[f"α_sp_{row}"]) * ARCSEC).mask(exclude)
    fig.add_trace(go.Scatter(x=pid_df.t_sec, y=equ_error, name=f"{EQU[i]} error",
        line=dict(color="white", width=1)), row=row, col=1)
    fig.add_trace(go.Scatter(x=pid_df.t_sec, y=topo_error, name=f"{TOPO[i]} error",
        line=dict(color="white", width=1)), row=row, col=2)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)
fig.update_xaxes(title_text="Time (s)", row=3, col=2)
fig.update_layout(height=900, width=1500, template="plotly_dark", hovermode="x unified",
    title="PID Equatorial (left) & Topocentric (right) Tracking Error",
    legend=dict(groupclick="toggleitem"))
fig.show()
# Az range / meridian-proximity is checked separately in the RA/PA anomaly-hunt section below,
# since absolute Az isn't on this chart anymore now that the right column is error, not SP/PV.


# Combined KF+PID timeline

Merges the two streams on nearest timestamp (they tick independently -- KF on every 518
message, PID on every control step -- so this aligns them within a short tolerance rather
than assuming a shared clock). Used by the "zoom into context" cells below.

In [ ]:
TOLERANCE = pd.Timedelta("100ms")

merged = pd.DataFrame()
if len(kf_df) and len(pid_df):
    merged = pd.merge_asof(
        pid_df.sort_values("timestamp"), kf_df.sort_values("timestamp"),
        on="timestamp", direction="nearest", tolerance=TOLERANCE,
        suffixes=("_pid", "_kf"),
    )
    merged["t_sec"] = (merged["timestamp"] - merged["timestamp"].iloc[0]).dt.total_seconds()
print(f"Merged samples: {len(merged)}")
merged.head()


# Anomaly Detection & Investigation

Two distinct failure signatures get their own detection + investigation pair below, instead
of one generic "PID reacted hard" bucket -- each has a different likely root cause and needs
different context to diagnose:

- **Spike Anomaly** -- a sudden, sharp PID reaction on one axis (a lone hard kick, or a short
  multi-tick "hump"). Usually the PID responding correctly to something real -- the question
  is *what* disturbed it.
- **Oscillation Anomaly** -- a sustained, periodic tracking error (multi-second period,
  survives several cycles) rather than a one-off transient. Originally reported near meridian
  transit as mirrored RA/PA error (issue #88), but detected here directly in motor-frame
  position error at **any** orientation, since a real periodic disturbance shows up there
  regardless of where on the sky the mount happens to be pointing.

(518 telemetry dropouts and sustained lag get their own dedicated notebook --
`analyse_comms.ipynb` -- with a fuller multi-file outage reconstruction than fits here; every
plot below still masks `near_gap`/`near_startup` ticks via `exclude`, same as before, so a
dropout's resettle transient doesn't get mistaken for a Spike or Oscillation anomaly.)

Each gets the same two-part treatment:
1. **Detection** -- an automatic scan producing a summary table of discrete *events* (not
   per-tick), with tunable thresholds documented inline, same convention as the rest of this
   notebook.
2. **Anomoly Analysis** -- an interactive explorer (dropdown, or step through with the slider) showing
   the detailed telemetry leading up to, through, and recovering from each detected event, so
   a flagged event can actually be diagnosed, not just counted.

In [ ]:
# ── Shared anomaly-hunter helpers (used by both anomaly sections below) ────────────────────
AXIS_COLORS = {1: ("royalblue", "deepskyblue"), 2: ("mediumseagreen", "palegreen"), 3: ("orange", "navajowhite")}
# (meas_color, state_color) per 1-based axis index -- consistent across every hunter below so
# "which axis is which colour" doesn't have to be relearned per section.

def event_shapes(t_start, t_end, t_center):
    """Dashed line at the peak/center tick (t=0) plus a shaded band over the event's actual
    span, both expressed relative to the event center so they match the traces' x-axis."""
    return [
        dict(type="line", xref="x", yref="paper", x0=0, x1=0, y0=0, y1=1,
             line=dict(color="red", width=1, dash="dash")),
        dict(type="rect", xref="x", yref="paper",
             x0=t_start - t_center, x1=t_end - t_center, y0=0, y1=1,
             fillcolor="red", opacity=0.08, line_width=0),
    ]

def window_slice(df, t_center, before_sec, after_sec=None):
    """[t_center - before_sec, t_center + after_sec] of a KF/PID dataframe, with t_sec rebased
    to 0 at t_center so different events' traces line up for direct comparison. after_sec
    defaults to before_sec (a symmetric window) -- Oscillation events pass distinct before/after."""
    if after_sec is None:
        after_sec = before_sec
    w = df[(df.t_sec > t_center - before_sec) & (df.t_sec < t_center + after_sec)].copy()
    w["t_rel"] = w.t_sec - t_center
    return w

def event_window(ev, window_sec):
    """(before, after) seconds either side of ev.t_sec to display. Uses the event's own
    window_before/window_after when the events_df provides them (Oscillation: sized to the
    event's real detected span plus a few periods of context, since a fixed +/-20s can run
    deep into masked/irrelevant data for a short fast oscillation, or barely cover a long
    slow one). Falls back to the shared symmetric window_sec otherwise (Spike: a fixed
    context window makes sense there, its events are always a couple of ticks wide)."""
    has_custom = "window_before" in ev.index and "window_after" in ev.index \
                 and pd.notna(ev.window_before) and pd.notna(ev.window_after)
    if has_custom:
        return float(ev.window_before), float(ev.window_after)
    return window_sec, window_sec

def clipped_shapes(ev, before, after):
    """event_shapes(), but with t_start/t_end first clipped to the actual displayed window.
    An event's own span can run well past the hunter's window (seen with Oscillation events,
    whose t_start/t_end come from a sparse detection grid -- Spike events never do, their
    span is a couple of ticks, which is why this only surfaced there). Plotly's autorange
    includes shapes as well as traces, so an unclipped rectangle reaching past the real data
    silently stretches the axis out to fit it, squeezing the actual traces into a fraction of
    the plot and making a real oscillation look artificially flat."""
    t_start = max(ev.t_start, ev.t_sec - before)
    t_end = min(ev.t_end, ev.t_sec + after)
    return event_shapes(t_start, t_end, ev.t_sec)

def single_axis_event_hunter(events_df, window_sec, title_prefix):
    """Interactive hunter for single-axis events (Spike, Oscillation): KF/PID context around
    each flagged event (see event_window() for how wide), one dropdown/slider entry per
    event, 4 panels (518 timing, theta position deviation, theta error, velocity components)
    for the one axis involved. Shared by both anomaly types below -- same navigation
    mechanism, just a different events_df and title."""
    fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
        specs=[[{}], [{"secondary_y": True}], [{}], [{}]],
        subplot_titles=["518 timing: receipt interval vs measurement_lag_s (s)",
                         "driver vitals events: lag_s (dots) -- CPU/Mem/Swap % hidden, own axis",
                         "\u03b8 error (arcsec)", "velocity components (arcsec/s)"],
        row_heights=[0.2, 0.3, 0.2, 0.3], vertical_spacing=0.06)

    if not len(events_df):
        print("No events flagged -- nothing to hunt.")
        return

    events = events_df.reset_index(drop=True)
    components = [("\u03c9_ff", "green"), ("\u03c9_kp", "magenta"), ("\u03c9_ki", "olive"), ("\u03c9_kd", "orange"), ("\u03c9_op", "cyan")]
    trace_spans = []
    true_trace_indices = []
    for ev_i, ev in events.iterrows():
        ax_idx = ev["axes"][0]   # bracket access -- ev.axes collides with pandas' own Series.axes property
        visible = (ev_i == 0)
        start = len(fig.data)

        before, after = event_window(ev, window_sec)
        kf_window = window_slice(kf_df, ev.t_sec, before, after) if len(kf_df) else pd.DataFrame()
        t_rel_kf = kf_window.t_rel if len(kf_window) else pd.Series(dtype=float)

        fig.add_trace(go.Scatter(x=t_rel_kf, y=(kf_window.gap_sec if len(kf_window) else []), name="518 receipt", visible=visible,
            mode="markers+lines", marker=dict(size=4, color="yellow"), line=dict(color="yellow", width=1)), row=1, col=1)
        lag_col = kf_window["measurement_lag_s"] if len(kf_window) and "measurement_lag_s" in kf_window.columns else pd.Series(dtype=float)
        fig.add_trace(go.Scatter(x=t_rel_kf, y=lag_col, name="measurement_lag_s", visible=visible,
            mode="markers+lines", marker=dict(size=4, color="orange"), line=dict(color="orange", width=1)), row=1, col=1)

        vitals_window = vitals_df[(vitals_df.t_sec > ev.t_sec - before) & (vitals_df.t_sec < ev.t_sec + after)].copy() \
            if "vitals_df" in globals() and len(vitals_df) else pd.DataFrame()
        if len(vitals_window):
            vitals_window["t_rel"] = vitals_window.t_sec - ev.t_sec
        VITALS_KIND_COLORS = {"518 lag detected": "yellow", "Heartbeat lag detected": "red"}
        for kind, color in VITALS_KIND_COLORS.items():
            sub = vitals_window[vitals_window.kind == kind] if len(vitals_window) else pd.DataFrame()
            hover = (sub.apply(lambda r: f"lag={r.lag_s:.2f}s  CPU={r.cpu_pct:.0f}% Mem={r.mem_pct:.0f}% Swap={r.swap_pct:.0f}%  "
                                          f"Threads={r.threads} NetDrops={r.net_dropin}/{r.net_dropout} NetErr={r.net_errin}/{r.net_errout}", axis=1)
                     if len(sub) else pd.Series(dtype=object))
            fig.add_trace(go.Scatter(x=(sub.t_rel if len(sub) else []), y=(sub.lag_s if len(sub) else []),
                name=kind, visible=visible, mode="markers", marker=dict(size=9, color=color, symbol="circle",
                line=dict(width=1, color="black")), text=hover, hovertemplate="%{text}<extra>" + kind + "</extra>"),
                row=2, col=1, secondary_y=False)
        # CPU/Mem/Swap % -- hidden by default, opt-in via legend, own (secondary) axis -- same
        # convention the removed theta_meas_true/theta_state_true traces used
        hidden_trace_idx = []
        for metric, color in [("mem_pct", "deepskyblue"), ("cpu_pct", "palegreen"), ("swap_pct", "orange")]:
            hidden_trace_idx.append(len(fig.data))
            fig.add_trace(go.Scatter(x=(vitals_window.t_rel if len(vitals_window) else []),
                y=(vitals_window[metric] if len(vitals_window) else []),
                name=metric, visible=False, mode="markers+lines", marker=dict(size=4, color=color),
                line=dict(color=color, width=1, dash="dot")), row=2, col=1, secondary_y=True)
        true_trace_indices.append(tuple(hidden_trace_idx))

        window = window_slice(pid_df, ev.t_sec, before, after)
        t_rel = window.t_rel
        error = (wrap_deg(window[f"\u03b8_pv_{ax_idx}"] - window[f"\u03b8_sp_{ax_idx}"]) * ARCSEC).mask(window.exclude)
        fig.add_trace(go.Scatter(x=t_rel, y=error, name="\u03b8 error", visible=visible,
            line=dict(color="white", width=1)), row=3, col=1)
        for key, color in components:
            fig.add_trace(go.Scatter(x=t_rel, y=(window[f"{key}_{ax_idx}"] * ARCSEC).mask(window.exclude),
                name=key, visible=visible, line=dict(color=color, width=1)), row=4, col=1)
        trace_spans.append((start, len(fig.data) - start))

    # Toggling trace visibility via an updatemenu does NOT itself make Plotly re-fit each
    # axis to whatever's now visible -- every axis keeps whatever range it was first given
    # (autorange computed for event 0's data at initial render), so later events can appear
    # to not update at all (a smaller-magnitude row looks flat/unchanged against a scale
    # sized for a bigger one) or look truncated (a differently-shaped window against a
    # locked-in range). Forcing autorange back on for every axis on every click fixes both --
    # confirmed by inspecting a real generated figure's layout, where shared_xaxes=True had
    # already linked every row's x-axis to row 4's (xaxis.matches="x4" etc.), which is exactly
    # what makes a stale range on any one of them look consistent across all four instead of
    # being an obvious mismatch.
    AXIS_AUTORANGE = {f"{ax}.autorange": True for ax in
                       ["xaxis", "xaxis2", "xaxis3", "xaxis4", "yaxis", "yaxis2", "yaxis3", "yaxis4", "yaxis5"]}

    n_total = len(fig.data)
    buttons = []
    for ev_i, ev in events.iterrows():
        start, count = trace_spans[ev_i]
        visible = [False] * n_total
        for j in range(start, start + count):
            visible[j] = True
        for idx in true_trace_indices[ev_i]:
            visible[idx] = False
        buttons.append(dict(
            label=ev.short_label, method="update",
            args=[{"visible": visible},
                  {"title": f"{title_prefix}: {ev.label}", "shapes": clipped_shapes(ev, *event_window(ev, window_sec)),
                   **AXIS_AUTORANGE}],
        ))

    slider_steps = list(buttons)   # same rich label as the dropdown (time + description), not a bare index -- a slider step needs to say what t is on its own, since moving the slider does not update the dropdown's own highlighted selection (a real Plotly limitation: the two controls share the same click/drag handler but neither is aware of the other's displayed state)

    fig.add_hline(y=0.2, row=1, col=1, line=dict(color="gray", width=1, dash="dot"))
    fig.update_yaxes(title_text="lag (s)", row=2, col=1, secondary_y=False)
    fig.update_yaxes(title_text="CPU/Mem/Swap %", row=2, col=1, secondary_y=True)
    fig.update_xaxes(title_text="Time relative to event peak (s)", row=4, col=1)
    fig.update_layout(height=1100, width=1300, template="plotly_dark", hovermode="x unified",
        title=f"{title_prefix}: {events.iloc[0].label}",
        shapes=clipped_shapes(events.iloc[0], *event_window(events.iloc[0], window_sec)),
        legend=dict(groupclick="toggleitem"),
        updatemenus=[dict(buttons=buttons, direction="down", x=1.0, xanchor="right", y=1.1, yanchor="top",
                           showactive=True, bgcolor="#2B2B2B", bordercolor="#777777", borderwidth=1,
                           font=dict(color="#AAA"))],
        sliders=[dict(active=0, x=0.0, len=0.88, pad=dict(t=60, b=10),
                       currentvalue=dict(prefix="Event (use \u2190/\u2192 or drag to step): ", font=dict(color="#EEEEEE", size=12)),
                       font=dict(color="#EEEEEE", size=10), bgcolor="#2B2B2B", bordercolor="#777777",
                       activebgcolor="#555555", steps=slider_steps)])
    fig.show()


MAX_HUNTER_EVENTS = 50   # cap on how many events single_axis_event_hunter() actually renders --
                          # each event adds ~13 traces (518 timing, vitals, theta error, 5 velocity
                          # components, hidden CPU/mem/swap), so a heavily-anomalous session (e.g. one
                          # spanning a real comms outage -- see analyse_comms.ipynb) can flag thousands
                          # of events and blow the hunter figure out to tens of thousands of traces,
                          # which is slow to build in Python and can hang the notebook trying to render
                          # it (confirmed: 1756 Spike + 1363 Oscillation events on a real capture took
                          # 54s + 35s just to build the figures, before the frontend even renders them).
                          # The summary table above each hunter still lists every flagged event -- only
                          # the interactive figure itself is capped, keeping the most severe events (by
                          # the caller's own severity_col) so triage isn't just chronological luck.

def top_events_for_hunter(events_df, severity_col, max_events=MAX_HUNTER_EVENTS):
    """Cap events_df to its max_events most severe rows (by abs(severity_col)), re-sorted back
    to chronological order for a natural dropdown/slider step-through. Prints a note when
    truncated; pass-through unchanged otherwise."""
    if len(events_df) <= max_events:
        return events_df
    top = events_df.reindex(events_df[severity_col].abs().sort_values(ascending=False).index)
    top = top.head(max_events).sort_values("t_sec").reset_index(drop=True)
    print(f"Showing the {max_events} most severe of {len(events_df)} flagged events in the hunter "
          f"below (ranked by |{severity_col}|) -- see the full table above for the rest.")
    return top

## Spike Anomaly: sudden PID reactions

Flags *events* -- runs of consecutive control ticks where `|\u03c9_kp|` on any axis sits far
above its local baseline -- a direct proxy for "the PID suddenly thought it was a long way
off target and kicked the motor hard", the kind of transient that could put a bend in a star
trail without ever crossing the log's `WARNING` lag thresholds. Uses a rolling median + MAD
(robust to the spikes themselves, unlike mean/stdev) for the per-tick threshold. Flagging
combines two rules so both shapes of disturbance get caught without drowning in one-off
noise: a single tick clearing the high `N_MAD_SPIKE` bar is an event on its own (a lone sharp
kick), while a lower `N_MAD_RUN` bar only counts with `MIN_RUN`+ consecutive ticks (a slower
multi-second "hump" that never spikes hard on any one tick). Consecutive flagged ticks on the
same axis within `CLUSTER_GAP_S` of each other are merged into a single event (start/end/peak)
rather than listed per-tick.

`MANUAL_SPIKE_EVENTS` below is an escape hatch for anything spotted by eye (e.g. in the
hunter) that the automatic thresholds don't clear -- add `(axis, t_sec)` and it's folded into
the same table/hunter, tagged `manual=True`, with no detection logic applied.

### Detection

In [ ]:
WINDOW = 51          # ticks (~10s at 200ms) for the rolling baseline
N_MAD_SPIKE = 8.0    # a single tick this far above baseline is an event on its own
N_MAD_RUN = 5.0      # a lower bar, but only counts with >=MIN_RUN consecutive ticks
MIN_RUN = 2          # minimum consecutive flagged ticks (same axis) for the N_MAD_RUN bar to count
CLUSTER_GAP_S = 1.5  # merge flagged ticks on the same axis into one event if within this many
                     # seconds of each other, so one multi-tick disturbance is one row, not N

MANUAL_SPIKE_EVENTS = [
#    ("M2", 1457.04),  # spotted by eye in the hunter around the M2 @ 1447.2s event -- a real
                       # ~2.4s sustained excursion, but its peak (~1.7 arcsec/s) never clears
                       # N_MAD_RUN against this stretch's low local baseline (~0.7), so automatic
                       # detection can't reach it without flooding the whole run with false positives
]

events = []
axis_stats = {}
for i, ax in enumerate(AXES):
    col = f"\u03c9_kp_{i+1}"
    series = (pid_df[col] * ARCSEC).abs()
    med = series.rolling(WINDOW, center=True, min_periods=WINDOW//2).median()
    mad = (series - med).abs().rolling(WINDOW, center=True, min_periods=WINDOW//2).median()
    robust_std = 1.4826 * mad
    axis_stats[ax] = (col, med)

    candidate = (series > med + N_MAD_RUN * robust_std) & (robust_std > 1e-6) & (~pid_df.exclude)
    flagged = pid_df.loc[candidate, ["t_sec", "timestamp"]].copy()
    flagged["val"] = pid_df.loc[candidate, col] * ARCSEC
    flagged["is_spike"] = (series[candidate] > (med[candidate] + N_MAD_SPIKE * robust_std[candidate])).values
    if not len(flagged):
        continue
    group = (flagged.t_sec.diff().fillna(0) > CLUSTER_GAP_S).cumsum()
    for _, g in flagged.assign(group=group).groupby("group"):
        if len(g) < MIN_RUN and not g.is_spike.any():
            continue
        peak = g.loc[g.val.abs().idxmax()]
        events.append(dict(axis=ax, axes=(i + 1,), t_sec=peak.t_sec, timestamp=peak.timestamp,
                            t_start=g.t_sec.min(), t_end=g.t_sec.max(), n_ticks=len(g),
                            omega_kp_arcsec_s=peak.val, local_median_arcsec_s=med.loc[peak.name],
                            manual=False,
                            short_label=f"{ax} @ {peak.t_sec:.1f}s",
                            label=f"{ax} @ {peak.t_sec:.1f}s ({peak.val:.1f} arcsec/s, {len(g)} ticks)"))

for ax, t_center in MANUAL_SPIKE_EVENTS:
    i = AXES.index(ax)
    col, med = axis_stats[ax]
    nearest = (pid_df.t_sec - t_center).abs().idxmin()
    val = pid_df[col][nearest] * ARCSEC
    events.append(dict(axis=ax, axes=(i + 1,), t_sec=pid_df.t_sec[nearest], timestamp=pid_df.timestamp[nearest],
                        t_start=pid_df.t_sec[nearest], t_end=pid_df.t_sec[nearest], n_ticks=1,
                        omega_kp_arcsec_s=val, local_median_arcsec_s=med[nearest], manual=True,
                        short_label=f"{ax} @ {t_center:.1f}s",
                        label=f"{ax} @ {t_center:.1f}s (manual)"))

spike_events_df = pd.DataFrame(events).sort_values("t_sec").reset_index(drop=True) if events else pd.DataFrame()
print(f"Flagged {len(spike_events_df)} Spike Anomaly event(s) across {len(AXES)} axes (telemetry-gap ticks excluded)")
spike_events_df.head(30)

### Anomoly Analysis
Pick any flagged Spike event from the dropdown (or step through with the slider) and see
+/-20s of position and velocity traces around it, with a shaded band marking the event's
actual start/end and a dashed marker at its peak tick (t=0). Time axis is relative to the
selected event's peak, so different events line up for direct comparison.

Four panels, top to bottom:
1. **518 timing (s)** -- raw receipt interval (yellow) alongside `measurement_lag_s` (orange;
   requires a capture with the `measurement_lag_s` KFLOG field -- older captures leave the
   panel empty).
2. **\u03b8_meas_dev / \u03b8_state_dev (arcsec)** -- deviation from a *backdated* theta_ref (see
   the KF section above), not raw position. Two more traces start **hidden** --
   `\u03b8_meas_true`/`\u03b8_state_true`, the absolute (non-backdated, own axis) values. Toggle
   via the legend to sanity-check a deviation spike against the real underlying measurement;
   they reset to hidden when you switch events.
3. **\u03b8 error (arcsec)** -- PID position error on the flagged axis.
4. **velocity components (arcsec/s)** -- \u03c9_ff/kp/ki/kd/op on the flagged axis.

In [ ]:
EVENT_WINDOW_SEC = 20.0
single_axis_event_hunter(top_events_for_hunter(spike_events_df, "omega_kp_arcsec_s"), EVENT_WINDOW_SEC, "Spike Anomaly")

## Oscillation Anomaly: extended deviation from steady tracking (any orientation)

Originally reported for issue #88 as RA/PA mirrored oscillation specifically near meridian
transit (Az≈178-180°, ~4-5s period, ±3-5"), and an earlier version of this detector tried to
prove genuine periodicity directly (windowed autocorrelation, searching for a lag with a
strong self-correlation peak). That turned out fragile in practice: a smooth one-directional
ramp/transient in ω_op correlates just as well at short lags as real periodicity does, for the
unrelated reason that it isn't jagged -- confirmed against two real flagged events in this
capture that were actually settling swings, not oscillation, and each fix for one shape of
false positive (requiring zero-crossings, then detrending before testing) only shifted the
failure mode rather than closing it.

Detection here is deliberately simpler: **ω_op should stay fairly steady while tracking**, so
this flags any *sustained* stretch where a motor's short-term variance or its net rate of
change (ramp) is elevated well above its own recent robust baseline -- using the same rolling
median + MAD approach as the Spike Anomaly detector above, just applied to ω_op instead of
ω_kp. This is a strict superset of "periodic oscillation" (a real oscillation shows elevated
variance too) without needing to prove periodicity at all, so it isn't fooled by a ramp the
way the correlation-based version was -- a ramp *is* a real deviation from steady tracking,
worth flagging on its own merits, not something to filter out.

Two things keep this from just re-discovering Spike Anomaly events under a new name:
- **`OSC_MIN_DURATION_S`** requires the flagged stretch to be meaningfully longer than a spike
  ever runs -- Spike Anomaly's own `CLUSTER_GAP_S`/`MIN_RUN` are tuned for a couple of ticks
  to a couple of seconds; this requires several seconds sustained.
- Ticks already inside a flagged **Spike Anomaly** event's own span (on the same axis, with a
  little padding) are excluded outright, so the same disturbance doesn't get reported under
  both anomaly types.

Az is still recorded against every flagged event purely as *context* (was this near meridian,
where issue #88 was first seen, or somewhere else entirely) -- not as a detection gate.

**Startup exclusion:** as elsewhere in this notebook, the first `STARTUP_EXCLUDE_SEC` of a run
are dropped before flagging -- the initial setpoint-establishment jump on a fresh driver start
is a real, large one-off ω_op transient, not a tracking disturbance, and would otherwise swamp
the variance filter.

### Detection


In [ ]:
OSC_STD_WINDOW_TICKS = 25        # ~5s -- window for measuring omega_op's short-term variance/ramp
OSC_BASELINE_WINDOW_TICKS = 751  # ~150s -- window for the robust baseline "normal steady tracking" is judged against.
                                   # Must be well wider than any real disturbance is likely to run, or the baseline
                                   # itself gets contaminated by the very thing being measured against it -- confirmed
                                   # empirically: at 50s, a real ~15s ramp occupied enough of its own baseline window
                                   # that the local median tracked the ramp almost exactly, masking it entirely.
OSC_N_MAD_STD = 5.0               # local variance must clear the baseline by this many robust-MAD multiples
OSC_N_MAD_DELTA = 5.0             # local net change (ramp) must clear the baseline by this many robust-MAD multiples
OSC_MIN_DURATION_S = 3.0          # minimum sustained duration to count as Extended Deviation -- deliberately
                                   # longer than Spike Anomaly's own CLUSTER_GAP_S/MIN_RUN ever runs
OSC_CLUSTER_GAP_S = 2.0           # merge flagged runs within this many seconds of each other into one event
OSC_SPIKE_EXCLUDE_PAD_S = 1.0     # padding either side of an already-flagged Spike event's own span, when
                                   # excluding it here -- the same disturbance shouldn't get reported twice
MERIDIAN_WINDOW = 5.0             # degrees either side of Az=180 -- reported purely as event context
OSC_HUNTER_PAD_FRACTION = 0.5     # extra context shown either side of the event's own span in the hunter,
                                   # as a fraction of the event's own duration
OSC_HUNTER_MIN_PAD_S = 5.0        # floor on that padding regardless of duration

def robust_baseline(series, window):
    """Rolling median + MAD -- same robust-baseline approach the Spike Anomaly detector above
    uses, applied here to omega_op's local variance/ramp instead of |omega_kp|."""
    med = series.rolling(window, center=True, min_periods=window // 2).median()
    mad = (series - med).abs().rolling(window, center=True, min_periods=window // 2).median()
    return med, 1.4826 * mad

n = len(pid_df)
az_arr = pid_df["\u03b1_pv_1"].to_numpy() if "\u03b1_pv_1" in pid_df.columns else np.full(n, np.nan)

osc_rows = []
for i, ax in enumerate(AXES):
    series = pid_df[f"\u03c9_op_{i+1}"] * ARCSEC

    local_std = series.rolling(OSC_STD_WINDOW_TICKS, center=True, min_periods=OSC_STD_WINDOW_TICKS).std()
    std_med, std_mad = robust_baseline(local_std, OSC_BASELINE_WINDOW_TICKS)
    std_trigger = (local_std > std_med + OSC_N_MAD_STD * std_mad) & (std_mad > 1e-9)

    local_delta = series.diff(OSC_STD_WINDOW_TICKS).abs()
    delta_med, delta_mad = robust_baseline(local_delta, OSC_BASELINE_WINDOW_TICKS)
    delta_trigger = (local_delta > delta_med + OSC_N_MAD_DELTA * delta_mad) & (delta_mad > 1e-9)

    # Ticks already claimed by a Spike Anomaly event on this same axis (padded a little) don't
    # also get reported here -- see markdown above.
    spike_mask = pd.Series(False, index=pid_df.index)
    for _, sp in spike_events_df[spike_events_df.axis == ax].iterrows():
        spike_mask |= (pid_df.t_sec >= sp.t_start - OSC_SPIKE_EXCLUDE_PAD_S) & \
                      (pid_df.t_sec <= sp.t_end + OSC_SPIKE_EXCLUDE_PAD_S)

    candidate = (std_trigger | delta_trigger) & (~pid_df.exclude) & (~spike_mask)
    flagged = pid_df.loc[candidate, ["t_sec", "timestamp"]].copy()
    if not len(flagged):
        continue
    flagged["local_std"] = local_std[candidate].to_numpy()
    flagged["local_delta"] = local_delta[candidate].to_numpy()
    flagged["trigger"] = np.where(std_trigger[candidate] & delta_trigger[candidate], "variance+ramp",
                          np.where(std_trigger[candidate], "variance", "ramp"))
    flagged["az"] = az_arr[candidate.to_numpy()]

    group = (flagged.t_sec.diff().fillna(0) > OSC_CLUSTER_GAP_S).cumsum()
    for _, g in flagged.assign(group=group).groupby("group"):
        t_start, t_end = g.t_sec.min(), g.t_sec.max()
        duration = t_end - t_start
        if duration < OSC_MIN_DURATION_S:
            continue
        peak = g.loc[g.local_std.idxmax()]
        pad = max(OSC_HUNTER_PAD_FRACTION * duration, OSC_HUNTER_MIN_PAD_S)
        osc_rows.append(dict(
            axis=ax, axes=(i + 1,),
            t_sec=peak.t_sec, t_start=t_start, t_end=t_end, duration_s=duration,
            window_before=(peak.t_sec - t_start) + pad, window_after=(t_end - peak.t_sec) + pad,
            n_ticks=len(g), trigger=peak.trigger,
            peak_std_arcsec_s=peak.local_std, peak_delta_arcsec_s=peak.local_delta,
            az_min=g.az.min(), az_max=g.az.max(),
            near_meridian=bool(((g.az - 180).abs() <= MERIDIAN_WINDOW).any()),
            short_label=f"{ax} @ {peak.t_sec:.1f}s",
            label=f"{ax} @ {peak.t_sec:.1f}s ({duration:.1f}s, {peak.trigger}, "
                  f"std={peak.local_std:.1f}, \u0394={peak.local_delta:.1f} arcsec/s)",
        ))

oscillation_events_df = pd.DataFrame(osc_rows).sort_values("t_sec").reset_index(drop=True) if osc_rows else pd.DataFrame()
if len(oscillation_events_df):
    n_meridian = int(oscillation_events_df.near_meridian.sum())
    print(f"Flagged {len(oscillation_events_df)} Extended Deviation event(s) across {len(AXES)} axes "
          f"(>= {OSC_MIN_DURATION_S:.0f}s sustained, telemetry-gap and Spike-claimed ticks excluded): "
          f"{n_meridian} near Az=180\u00b1{MERIDIAN_WINDOW:.0f}\u00b0 (meridian), "
          f"{len(oscillation_events_df) - n_meridian} elsewhere.")
    display(oscillation_events_df)
else:
    print("No Oscillation Anomaly events flagged at this threshold -- try lowering OSC_N_MAD_STD / "
          "OSC_N_MAD_DELTA or OSC_MIN_DURATION_S.")


### Anomoly Analysis

Same interactive explorer as the Spike Anomaly hunter above, over the flagged Extended
Deviation events instead -- the dropdown label shows how long the deviation lasted, which
criterion tripped it (variance, ramp, or both), and the peak values for quick triage before
diving into the traces.

In [ ]:
single_axis_event_hunter(top_events_for_hunter(oscillation_events_df, "peak_std_arcsec_s"), EVENT_WINDOW_SEC, "Oscillation Anomaly")

# Notes